# GOES-18 ABI Ash RGB — Full Disk, then a domain

The Ash RGB uses infrared brightness-temperature differences, so it works day and night.

Everything is in this notebook: where the files come from, the domain,
and the whole plotting code. Change any of it and re-run — nothing is hidden in
a library.

**Steps:** point at your folder → plot the complete scan → set your domain →
plot that domain.


## Setup

In [ ]:
import glob
import math
import warnings
from pathlib import Path

import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pyresample import create_area_def
from satpy import Scene
from satpy.enhancements.enhancer import get_enhanced_image
from shapely.geometry import box

warnings.filterwarnings("ignore")


In [ ]:
# ---------------------------------------------------------------
# Helper kept in the notebook so you can change it: coastline segments
# clipped to a lon/lat box. Change "10m" to "50m" for a coarser, faster line.
# ---------------------------------------------------------------
def coastlines(bbox, resolution="10m"):
    west, south, east, north = bbox[0], bbox[1], bbox[2], bbox[3]
    clip = box(west, south, east, north)
    segments = []
    feature = cfeature.NaturalEarthFeature("physical", "coastline", resolution)
    for geometry in feature.intersecting_geometries([west, east, south, north]):
        piece = geometry.intersection(clip)
        for line in (piece.geoms if hasattr(piece, "geoms") else [piece]):
            if not line.is_empty and line.length > 0:
                x, y = line.xy
                segments.append((np.asarray(x), np.asarray(y)))
    return segments


def lon_label(value, _pos=None):
    v = ((value + 180) % 360) - 180
    return f"{abs(v):g}\u00b0{'E' if v >= 0 else 'W'}"


def lat_label(value, _pos=None):
    return f"{abs(value):g}\u00b0{'N' if value >= 0 else 'S'}"


In [ ]:
# ---------------------------------------------------------------
# STYLE -- change anything here and re-run the plot cells
# ---------------------------------------------------------------
COAST_COLOUR = "red"        # coastline colour
COAST_WIDTH = 0.8           # coastline line width
COAST_RES = "10m"           # "10m", "50m" or "110m"

GRID_COLOUR = "white"       # graticule colour
GRID_ALPHA = 0.45
GRID_STYLE = "--"
GRID_WIDTH = 0.5

MARKER_LON, MARKER_LAT = -163.9711, 54.7554   # Shishaldin; None to hide
MARKER_COLOUR = "red"
MARKER_SIZE = 10

FIG_WIDTH = 13.5            # inches
DPI = 160


## 1. Your files

In [ ]:
# ---------------------------------------------------------------
# 1. YOUR FILES  -- point this at the folder where your files are
# ---------------------------------------------------------------
DATA_DIR = Path("../data/ash_rgb")

files = sorted(glob.glob(str(DATA_DIR / "*.nc")))
print(f"folder: {DATA_DIR}")
print(f"{len(files)} file(s) found:")
for name in files:
    print("   ", Path(name).name)

# If the folder is empty, this fetches the example scan so the notebook can
# run. Delete this block once you use your own files.
if not files:
    import sys
    sys.path.insert(0, "..")
    from examples.glm import abi_scan_keys, download_abi
    from datetime import datetime
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    moment = datetime.fromisoformat("2023-10-03 19:00")
    keys = abi_scan_keys("noaa-goes18", "ABI-L1b-RadF", moment, ("C11", "C13", "C14", "C15",))
    files = sorted(download_abi("noaa-goes18", keys, DATA_DIR))
    print(f"downloaded {len(files)} file(s)")

# All channels must come from the SAME scan.


## 2. Build the composite

G18 needs `C11`, `C13`, `C14`, `C15`.

In [ ]:
COMPOSITE = "ash"
TITLE = "Ash RGB"

OUT_FULL = Path("../output/ash_rgb_full.png")
OUT_DOMAIN = Path("../output/ash_rgb_domain.png")
OUT_FULL.parent.mkdir(parents=True, exist_ok=True)

scene = Scene(reader="abi_l1b", filenames=files)
# generate=False keeps the channels as plain 2-D arrays. The RGB is built later,
# after resampling -- an already-built 3-D composite cannot be aggregated.
scene.load([COMPOSITE], generate=False)
print("loaded channels for", COMPOSITE)


## 3. The complete scan

In [ ]:
# The composite is built on the coarsest common grid and reduced in size,
# otherwise a Full Disk RGB is very heavy (C02 alone is 21696 x 21696).
# The reduction factor must divide the 0.5 / 1 / 2 km ABI grids evenly, so use
# a power of two. Raise TARGET_PIXELS for a sharper (heavier) overview.
TARGET_PIXELS = 1200
area = scene.coarsest_area()
step = 2 ** max(0, math.ceil(math.log2(max(area.width, area.height) / TARGET_PIXELS)))
print(f"reducing by {step}x -> about {area.width // step} px")
small = scene.resample(area.aggregate(x=step, y=step), resampler="native")
try:
    small[COMPOSITE]
except KeyError:
    small.load([COMPOSITE], generate=True)

picture = get_enhanced_image(small[COMPOSITE]).pil_image()

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(picture, origin="upper")
ax.set_title("G18 - " + TITLE, loc="left")
ax.set_title(f"{small[COMPOSITE].attrs['start_time']:%Y/%m/%d %H:%M} UTC", loc="right")
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
fig.savefig(OUT_FULL, dpi=DPI)
plt.close(fig)
display(Image(filename=str(OUT_FULL)))


## 4. Your domain

Write the four numbers yourself: `(min_lon, min_lat, max_lon, max_lat)`.


In [ ]:
DOMAIN = (-169.0, 52.5, -159.0, 57.0)
RESOLUTION = 0.02

print("domain:", DOMAIN)


## 5. Plot your domain

In [ ]:
# ---------------------------------------------------------------
# 5. PLOT YOUR DOMAIN -- all of the drawing is here, change what you like
# ---------------------------------------------------------------
area = create_area_def(
    "domain", {"proj": "longlat", "datum": "WGS84"},
    area_extent=DOMAIN, resolution=(RESOLUTION, RESOLUTION), units="degrees",
)
local = scene.resample(area)
try:
    local[COMPOSITE]
except KeyError:
    local.load([COMPOSITE], generate=True)

picture = get_enhanced_image(local[COMPOSITE]).pil_image()
west, south, east, north = DOMAIN

fig, ax = plt.subplots(figsize=(FIG_WIDTH, FIG_WIDTH * (north - south) / (east - west) * 0.92))

ax.imshow(picture, extent=(west, east, south, north), origin="upper",
          aspect="auto", zorder=0)

ax.grid(color=GRID_COLOUR, alpha=GRID_ALPHA, ls=GRID_STYLE, lw=GRID_WIDTH, zorder=1)

for line_x, line_y in coastlines(DOMAIN, COAST_RES):
    ax.plot(line_x, line_y, color=COAST_COLOUR, lw=COAST_WIDTH, zorder=4)

if MARKER_LON is not None and west <= MARKER_LON <= east and south <= MARKER_LAT <= north:
    ax.plot(MARKER_LON, MARKER_LAT, "^", color=MARKER_COLOUR, ms=MARKER_SIZE, zorder=6)

ax.set_xlim(west, east); ax.set_ylim(south, north)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lon_label))
ax.yaxis.set_major_formatter(plt.FuncFormatter(lat_label))
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_title("G18 - " + TITLE, loc="left")
ax.set_title(f"{local[COMPOSITE].attrs['start_time']:%Y/%m/%d %H:%M} UTC", loc="right")

fig.tight_layout()
fig.savefig(OUT_DOMAIN, dpi=DPI)
plt.close(fig)
display(Image(filename=str(OUT_DOMAIN)))


## Reading the colours

Ash appears red to magenta, SO₂ can appear bright green, a mixture yellow. Colours depend on cloud height, opacity and moisture, so read it qualitatively.